# HW2 HSE NLP — T4 inference (greedy, batch, KV-cache)

**Model:** from-scratch encoder–decoder Transformer, checkpoint after first epoch (step 57502). **Goal:** generate all 1919 test solutions on one **NVIDIA T4** within 180 seconds.

This notebook uses a prepared `hw2_t4_bundle.zip` exported from Vast.ai by `hw2_export_t4.py`. No training, no dataset download, no additional targets.

Run on **Google Colab → Runtime → Change runtime type → T4 GPU**. Run cells sequentially. Measure actual inference time; the notebook does not claim the 180-second limit is met without a T4 result.

In [ ]:
import sys, subprocess, importlib.util
import torch
print('Python:',sys.version.split()[0])
print('PyTorch:',torch.__version__,'CUDA:',torch.version.cuda)
assert torch.cuda.is_available(), 'Choose GPU runtime in Colab first'
print('GPU:',torch.cuda.get_device_name(0))
print('VRAM GB:',round(torch.cuda.get_device_properties(0).total_memory/1e9,2))
assert 'T4' in torch.cuda.get_device_name(0), 'This must run on a single NVIDIA T4 for the assignment speed proof'
if importlib.util.find_spec('tokenizers') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet','--no-deps','tokenizers>=0.20,<1'])
from tokenizers import Tokenizer
print('Dependencies OK')

## 1. Upload export ZIP

Run `hw2_export_t4.py` on Vast.ai, download `exports/hw2_t4_bundle.zip`, then upload it here when prompted. (If already uploaded to `/content/`, the cell uses that file.)

In [ ]:
from pathlib import Path
import zipfile, hashlib, json, time, os, sys

ZIP=Path('/content/hw2_t4_bundle.zip')
if not ZIP.exists():
    from google.colab import files
    print('Upload hw2_t4_bundle.zip from Vast.ai')
    files.upload()
assert ZIP.is_file(), 'File not found in /content/'
ROOT=Path('/content/hw2_t4')
ROOT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(ZIP,'r') as z:
    expected={'hw2_model.py','model_fp16.pt','test_inputs.json','tokenizer.json','manifest.json'}
    assert expected.issubset(set(z.namelist()))
    z.extractall(ROOT)
manifest=json.loads((ROOT/'manifest.json').read_text())
def sha256(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''): h.update(b)
    return h.hexdigest()
for name, expected in manifest['sha256'].items():
    got=sha256(ROOT/name)
    assert got==expected, f'Hash mismatch for {name}'
print('All hashes verified. Step:',manifest['step'])
print('Test rows:',manifest['test_rows'])

## 2. Load FP16 model and pretokenized test inputs

Test `row_id` is retained and all outputs are returned in its original order. No test inputs are truncated. The model uses FP16 on T4, shared embeddings and cached autoregressive decoding.

In [ ]:
import gc
sys.path.insert(0,str(ROOT))
from hw2_model import CodeTransformer, ModelConfig, PAD, EOS

test_rows=json.loads((ROOT/'test_inputs.json').read_text(encoding='utf-8'))
assert len(test_rows)==1919
assert len({r['row_id'] for r in test_rows})==1919
assert all(r['input_ids'][-1]==EOS for r in test_rows)
rows=sorted(test_rows,key=lambda r:r['row_id'])
tok=Tokenizer.from_file(str(ROOT/'tokenizer.json'))
assert sha256(ROOT/'tokenizer.json')==manifest['tokenizer_sha256']

payload=torch.load(ROOT/'model_fp16.pt',map_location='cpu',weights_only=True)
assert payload['step']==57502
net=CodeTransformer(ModelConfig(**payload['model_config'])).half()
incompat=net.load_state_dict(payload['model'],strict=False)
assert set(incompat.missing_keys)=={'head.weight'} and not incompat.unexpected_keys, incompat
assert net.emb.weight.data_ptr()==net.head.weight.data_ptr()
net=net.cuda().eval()
del payload
gc.collect()
print('Model loaded. Precision:',net.emb.weight.dtype)
print('Parameters:',sum(p.numel() for p in net.parameters()))
print('Test:',len(rows),'rows')

## 3. Baseline generation benchmark on 256 representative test inputs

Tests the **unchanged** `CodeTransformer.generate`, so there is no inference-algorithm modification. `MAX_NEW_TOKENS=384` is the configured generation limit; a missing EOS is recorded rather than silently declared complete. We will benchmark batch sizes and then run all test rows once.

In [ ]:
import time

MAX_NEW_TOKENS=384

def predict(rows_to_run, batch_size=32, max_new_tokens=MAX_NEW_TOKENS):
    indexed=sorted(enumerate(rows_to_run),key=lambda p:len(p[1]['input_ids']))
    generated=[None]*len(rows_to_run)
    torch.cuda.synchronize()
    t0=time.perf_counter()
    with torch.inference_mode():
        for s in range(0,len(indexed),batch_size):
            group=indexed[s:s+batch_size]
            n=max(len(r['input_ids']) for _,r in group)
            src=torch.full((len(group),n),PAD,dtype=torch.long)
            for j,(_,r) in enumerate(group):
                ids=r['input_ids']
                src[j,:len(ids)]=torch.tensor(ids,dtype=torch.long)
            out=net.generate(src.cuda(),max_new_tokens=max_new_tokens).cpu().tolist()
            for j,(original_index,_) in enumerate(group):
                generated[original_index]=out[j]
    torch.cuda.synchronize()
    return generated,time.perf_counter()-t0

# Source-length stratified sample across all test questions, not just the shortest.
ranked=sorted(rows,key=lambda r:len(r['input_ids']))
sample=[ranked[min(len(ranked)-1,int((i+.5)*len(ranked)/256))] for i in range(256)]
# Warm up once, excluding initialization effects from timed benchmarks.
_, warmup_seconds=predict(sample[:16],batch_size=16,max_new_tokens=384)
print('Warmup seconds:',round(warmup_seconds,3))
benchmark={}
for bs in (16,32,64):
    outputs,seconds=predict(sample,batch_size=bs)
    benchmark[bs]={'seconds_256':round(seconds,3),'estimated_1919_seconds':round(seconds*1919/256,1),
                  'eos_rate':round(sum(EOS in x for x in outputs)/len(outputs),4)}
    print('Batch size:',bs,benchmark[bs],flush=True)
print('These estimates are not a substitute for full T4 timing.')

## 4. Generate all 1919 solutions and measure T4 time

Uses the fastest sampled batch size. Measures **complete all-test generation**, not the RTX 5090 extrapolation. Results are ordered by original `row_id`. If the time is above 180 seconds, it **does not meet the rule**, and optimization must be done before submitting.

In [ ]:
BATCH_SIZE=min(benchmark,key=lambda bs:benchmark[bs]['seconds_256'])
print('Chosen batch size:',BATCH_SIZE)
all_outputs, generation_seconds=predict(rows,batch_size=BATCH_SIZE)
eos_count=sum(EOS in out for out in all_outputs)
print('T4 GENERATION RESULT')
print(json.dumps({'gpu':torch.cuda.get_device_name(0), 'step':57502,
                  'test_rows':len(rows),'max_new_tokens':MAX_NEW_TOKENS,
                  'batch_size':BATCH_SIZE,'generation_seconds':round(generation_seconds,3),
                  'examples_per_second':round(len(rows)/generation_seconds,3),
                  'eos':eos_count,'missing_eos':len(rows)-eos_count,
                  'within_180_seconds':generation_seconds<=180},indent=2))

## 5. Decode and save responses

Output is written as `predictions.jsonl` containing `row_id` and generated Python code. **This is an intermediate unambiguous format; check the Telegram bot's required submission format before sending.** `predictions.json` is also saved as a simple list ordered by `row_id`.

In [ ]:
out=[]
for r,ids in zip(rows,all_outputs):
    finished=EOS in ids
    cut=ids[:ids.index(EOS)] if finished else ids
    code_text=tok.decode(cut,skip_special_tokens=True)
    out.append({'row_id':int(r['row_id']),'code':code_text,'eos':bool(finished)})
assert len(out)==1919 and len({r['row_id'] for r in out})==1919

jsonl=Path('/content/predictions.jsonl')
with jsonl.open('w',encoding='utf-8') as f:
    for r in out:
        f.write(json.dumps({'row_id':r['row_id'],'code':r['code']},ensure_ascii=False)+'\n')
Path('/content/predictions.json').write_text(json.dumps([r['code'] for r in out],ensure_ascii=False),encoding='utf-8')
summary={'checkpoint_step':57502,'model':'from-scratch encoder-decoder',
         'device':torch.cuda.get_device_name(0),'batch_size':BATCH_SIZE,
         'max_new_tokens':MAX_NEW_TOKENS,'test_rows':len(rows),
         'generation_seconds':round(generation_seconds,3),'eos':eos_count,
         'within_180_seconds':generation_seconds<=180}
Path('/content/t4_benchmark.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print('Saved:',jsonl)
print('Sample first code:',out[0]['code'][:500])
print('Missing EOS:',len(rows)-eos_count)
print('Time:',round(generation_seconds,3),'s')
if generation_seconds>180:
    print('WARNING: T4 limit NOT MET. Do not claim compliance!')
else:
    print('Generation completed within 180 seconds on one NVIDIA T4.')

## 6. Download results

Use these intermediate files to prepare the final format after checking the Telegram bot instructions. Retain `t4_benchmark.json` and Colab output for the mandatory report.

In [ ]:
from google.colab import files
files.download('/content/t4_benchmark.json')
files.download('/content/predictions.jsonl')